<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/Thesis_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install the Agents SDK

!pip install openai-agents

In [ ]:
from google.colab import userdata
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# **Create agents**

In [ ]:
from agents import Agent

**First Agent**

In [ ]:
qa_agent = Agent(
    name = 'QA Agent',
    handoff_description = 'Agent for general question answering',
    instructions = '',
    model = ''
)

**Second Agent**

In [ ]:
translator_agent = Agent(
    name = 'Translator Agent',
    handoff_description = 'Agent for translating Natural Language to Sparql',
    instructions = 'You will convert natural language query into sparql',
    model = ''
    handoffs = [qa_agent]
)

**Third Agent**

In [ ]:
sparql_agent = Agent(
    name = 'SPARQL Agent',
    handoff_description = 'Agent for executing SPARQL queries',
    instructions = '',
    model = ''
    handoffs = [translator_agent]
)

Handoffs are sub-agents that the agent can delegate to. You provide a list of handoffs, and the agent can choose to delegate to them if relevant. This is a powerful pattern that allows orchestrating modular, specialized agents that excel at a single task.

**Fourth Agent**

In [ ]:
response_selector_agent = Agent(
    name = 'Response Selector Agent',
    handoff_description = 'Decides which responses to use or combines them',
    instructions = '',
    handoffs = [qa_agent, sparql_agent] # define an inventory of outgoing handoff options that the agent can choose from to decide how to make progress on their task
    model = ''
)

**Fifth Agent**

In [ ]:
evaluator_agent = Agent(
    name = 'Evaluator Agent',
    handoff_description = "Evaluates agents' responses based on algorithms",
    instructions = '',
    handoffs = [qa_agent, sparql_agent, response_selector_agent] # define an inventory of outgoing handoff options that the agent can choose from to decide how to make progress on their task
    model = ''
)

# **Run the agent orchestration**

Check that the workflow runs and the triage agent correctly routes between the two specialist agents.

In [ ]:
from agents import Runner

async def main():
    result = await Runner.run(triage_agent, "What is the capital of France?")
    print(result.final_output)

# **Add a guardrail**

You can define custom guardrails to run on the input or output.

In [ ]:
from agents import GuardrailFunctionOutput, Agent, Runner
from pydantic import BaseModel

class HomeworkOutput(BaseModel):
    is_homework: bool
    reasoning: str

guardrail_agent = Agent(
    name="Guardrail check",
    instructions="",
    output_type=HomeworkOutput,
)

async def homework_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(HomeworkOutput)
    return GuardrailFunctionOutput(
        output_info=final_output,
        tripwire_triggered=not final_output.is_homework,
    )

# **Put it all together**

In [ ]:
from agents import Agent, InputGuardrail, GuardrailFunctionOutput, Runner
from pydantic import BaseModel
import asyncio

class HomeworkOutput(BaseModel):
    is_homework: bool
    reasoning: str

guardrail_agent = Agent(
    name="Guardrail check",
    instructions="",
    output_type=HomeworkOutput,
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="",
    instructions="",
)

history_tutor_agent = Agent(
    name="",
    handoff_description="",
    instructions="",
)


async def homework_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(HomeworkOutput)
    return GuardrailFunctionOutput(
        output_info=final_output,
        tripwire_triggered=not final_output.is_homework,
    )

triage_agent = Agent(
    name="Triage Agent",
    instructions="",
    handoffs=[history_tutor_agent, math_tutor_agent],
    input_guardrails=[
        InputGuardrail(guardrail_function=homework_guardrail),
    ],
)

async def main():
    result = await Runner.run(triage_agent, "who was the first president of the united states?")
    print(result.final_output)

    result = await Runner.run(triage_agent, "what is life")
    print(result.final_output)

if __name__ == "__main__":
    asyncio.run(main())